In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import os
os.chdir(r'C:\Users\Lenovo\Desktop\Diabetes_Classifier\notebooks')
os.chdir("..")
print(os.getcwd())

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [15]:
from modeling.XGBoost import XGBoost
from modeling.Ada import Ada
from modeling.MLP import MLP
from modeling.RandomForest import RandomForest
from modeling.Ensemble import LR,StackingEnsemble
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [16]:
PATH="data/clean/clean.csv"
df=pd.read_csv(PATH)
df.head()
df.columns = df.columns.str.strip()

In [17]:
xgb=XGBoost()
xgb_params={'n_estimators': 7800, 'max_depth': 13, 'learning_rate': 0.03585979413065538, 'gamma': 0.00025046646096832056, 'min_child_weight': 8, 'reg_alpha': 9.979812055651985, 'reg_lambda': 0.041477402642739976, 'subsample': 0.9999355054935987, 'colsample_bytree': 0.8896668801909641, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)

rf=RandomForest()
rf_params={'n_estimators': 1200, 'max_depth': 6, 'min_samples_split': 18, 'min_samples_leaf': 9, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
rf.set_params(rf_params)

ada=Ada()
ada_params={'n_estimators': 364, 'learning_rate': 0.6133741065916875}
ada.set_params(ada_params)

mlp=MLP()
mlp_params={'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 6.120954197759074e-05, 'learning_rate_init': 0.006507372434299954, 'batch_size': 64, 'max_iter': 900, 'early_stopping': True, 'random_state': 42}
mlp.set_params(mlp_params)


In [18]:
meta_x_model=LR()
params={'penalty': 'l2', 'l1_ratio': 0.377184927025366, 'C': 0.3926479245177274, 'class_weight': 'balanced','n_jobs':-1}
meta_x_model.set_params(params)

In [19]:
stack=StackingEnsemble([xgb,rf,mlp,ada],meta_x_model,df,5,'diagnosis',['XGBoost,RandomForest,Ada'])


      age  bmi  chol    tg       hdl       ldl    cr       bun    lipids  \
4075   53   26  5.56  2.04  0.970000  3.660000  72.6  3.780000  0.265027   
3181   37   23  5.78  0.83  1.270000  4.090000  79.2  4.820000  0.310513   
1513   59   33  8.52  5.18  4.860753  4.860753  56.8  5.690000  1.000000   
4318   45   27  4.96  2.48  1.060000  2.720000  72.3  4.380000  0.389706   
3167   70   26  4.05  0.68  4.860753  4.860753  67.9  4.860753  1.000000   
...   ...  ...   ...   ...       ...       ...   ...       ...       ...   
3027   56   22  5.59  0.66  4.860753  4.860753  69.1  5.170000  1.000000   
2929   78   27  4.87  1.40  1.050000  3.030000  47.4  5.600000  0.346535   
3976   50   24  2.00  0.80  0.600000  1.000000  74.0  5.000000  0.600000   
4108   72   24  4.74  1.71  1.450000  2.590000  69.2  5.760000  0.559846   
949    42   27  4.72  3.90  1.510000  2.070000  84.0  4.600000  0.729469   

      Age_x_BMI  HDL_x_LDL    BMI/LDL  BMI/HDL+LDL    bun_x_cr  chol/ldl  
4075       1

In [20]:
stack.fit_base_models()

In [21]:
stack.fit_meta_model()

XGBoost fold 1/5 done
XGBoost fold 2/5 done
XGBoost fold 3/5 done
XGBoost fold 4/5 done
XGBoost fold 5/5 done
RandomForest fold 1/5 done
RandomForest fold 2/5 done
RandomForest fold 3/5 done
RandomForest fold 4/5 done
RandomForest fold 5/5 done
MLP fold 1/5 done
MLP fold 2/5 done
MLP fold 3/5 done
MLP fold 4/5 done
MLP fold 5/5 done
Ada fold 1/5 done
Ada fold 2/5 done
Ada fold 3/5 done
Ada fold 4/5 done
Ada fold 5/5 done
      XGBoost_pred  RandomForest_pred  MLP_pred  Ada_pred
1215      0.323270           0.455491  0.385598  0.495663
19        0.108783           0.159017  0.043013  0.369981
2093      0.033843           0.040798  0.027734  0.377956
668       0.064087           0.053313  0.037806  0.407496
218       0.182232           0.253950  0.271230  0.461056
...            ...                ...       ...       ...
4426      0.891439           0.774983  0.884366  0.560844
466       0.231532           0.285293  0.216142  0.406356
3092      0.197032           0.297926  0.292274  0.46

In [22]:
m=stack.cv(5)
print(m)
m.out()

ModelMetrics(accuracy=array([0.82825822, 0.81851401, 0.83069428, 0.82195122, 0.82560976]), balanced_accuracy=array([0.82427309, 0.8163523 , 0.83518289, 0.82328712, 0.81918073]), precision=array([0.76119403, 0.74344023, 0.74450549, 0.74011299, 0.76452599]), f1=array([0.78341014, 0.77389985, 0.7958884 , 0.78208955, 0.77760498]), recall=array([0.80696203, 0.80696203, 0.85488959, 0.82911392, 0.79113924]), roc_auc=array([0.90989472, 0.90761374, 0.92520405, 0.91797895, 0.9074995 ]))
Accuracy: 0.8250054959745701, std: 0.004353747685608437
Precision: 0.7507557491652934, std: 0.010044182693977066
F1: 0.7825785829654129, std: 0.007458405256324119
Recall: 0.8178133610190471, std: 0.022133057409445208
Balanced Accuracy:0.8236552267927234, std: 0.006430523072245744
ROC-AUC: 0.9136381912894513, std: 0.0069368923281369005


In [23]:
stack.measure()

{'accuracy': 0.8304093567251462, 'balanced': 0.8301466596394733, 'precision': 0.7652370203160271, 'f1': 0.795774647887324, 'recall': 0.8288508557457213, 'roc_auc': 0.9275419749319405}


In [39]:
input=pd.DataFrame({'age':[80],'gender':[1],'bmi':[25],'chol':[5.5],'tg':[5.8],'hdl':[4.8],'ldl':[4.2],'cr':[64],'bun':[4.8]})
result=stack.predict_single(input)
print(result)

You have diabetes!


In [40]:
PATH="models/StackEnsemble.pkl"
stack.save(PATH)

Stacking ensemble saved to models/StackEnsemble.pkl
